# Step 3 V2 - positive cross-fitted eventless counterfactuals and literal PDF weights

This notebook implements Exercise 3 using only frozen V2 Step 1/2 model artifacts plus the audited raw event calendars. An **eventless training row** is a usable model day containing no canonical target event and lying outside the next `P` model-day rows whose lag blocks could contain that event. Predictions are positive log-model conditional medians.

The literal occurrence weight is always `omega_i = actual_i / counterfactual_i`. An empirical-null-normalised statistic is exported separately as `omega_null_normalized`; it is never called the PDF weight. Offset 0 is pre-specified primary. Offsets -1/+1 are secondary profiles, with circular-shift max-statistic inference for any across-offset search.

In [1]:
from pathlib import Path
import os,json
import numpy as np
import pandas as pd
import statsmodels.api as sm
from scipy import stats
from statsmodels.stats.multitest import multipletests
from IPython.display import display

SEED=16017; rng=np.random.default_rng(SEED); ALPHA=.05; N_PERM=499; N_BOOT=999; MIN_TRAIN=200
BASE=Path(os.environ.get("PIPELINE_BASE",Path.cwd())).resolve(); IN1=BASE/"step1_results_v2"; IN2=BASE/"step2_results_v2"; OUT=BASE/"step3_results_v2"; OUT.mkdir(parents=True,exist_ok=True)
panel=pd.read_csv(IN1/"step1_panel_v2.csv",parse_dates=["model_day","session_start_utc","session_end_utc"])
dec=json.loads((IN1/"frozen_decisions.json").read_text()); s2=json.loads((IN2/"step2_summary.json").read_text())
assert s2["split_date"]==dec["split_date"] and s2["all_forecasts_positive"]
P=int(dec["primary_lag_order"]); TIMING=dec["chosen_eq2_timing"]; WEEKDAY=bool(dec["weekday_adjustment"])
EVENTS=BASE/"Data/interim/g10_usd_jpy_events_audited.csv"; BOJ=BASE/"Data/interim/boj_calendar_audited.csv"
for p in (EVENTS,BOJ): assert p.exists()
TARGETS=["FOMC Rate Decision (Upper Bound)","BOJ Target Rate","Change in Nonfarm Payrolls","CPI MoM","CPI YoY","Natl CPI YoY","Tokyo CPI Ex-Fresh Food YoY"]
print("frozen",TIMING,P,WEEKDAY,dec["eq2_blocks_retained"],dec["eq3_blocks_retained"])

frozen advance 5 True ['iv', 'rv'] ['iv']


## Event timestamp and session mapping audit

Mapping is exact-label, ISO-8601 parsed, nanosecond `searchsorted`, DST-aware, and reconciled so every source row is before, after, inside-unmapped, or mapped. BOJ calendar counts below are generated, never embedded in prose.

In [2]:
g=pd.read_csv(EVENTS,low_memory=False); g["t"]=pd.to_datetime(g.event_datetime_utc,utc=True,format="ISO8601",errors="coerce"); assert g.t.notna().all()
lab=pd.DataFrame({"event":TARGETS,"rows_in_file":[int(g.event.eq(x).sum()) for x in TARGETS]}); assert lab.rows_in_file.gt(0).all()
ev=g[g.event.isin(TARGETS)].copy(); E=panel.session_end_utc.astype("int64").to_numpy(); S=panel.session_start_utc.astype("int64").to_numpy(); T=ev.t.astype("int64").to_numpy()
raw=np.searchsorted(E,T,side="right"); idx=np.clip(raw,0,len(panel)-1); ev["mapped"]=(raw<len(panel))&(S[idx]<=T); ev["panel_row"]=np.where(ev.mapped,idx,-1); ev["model_day"]=pd.NaT; ev.loc[ev.mapped,"model_day"]=panel.model_day.to_numpy()[idx[ev.mapped]]
before=(~ev.mapped)&(T<S[0]); after=(~ev.mapped)&(T>=E[-1]); inside=(~ev.mapped)&~before&~after
recon=pd.DataFrame({"rows_in_file":ev.groupby("event").size(),"before":ev[before].groupby("event").size(),"after":ev[after].groupby("event").size(),"inside_unmapped":ev[inside].groupby("event").size(),"mapped":ev[ev.mapped].groupby("event").size(),"distinct_model_days":ev[ev.mapped].groupby("event").panel_row.nunique()}).fillna(0).astype(int).reindex(TARGETS)
assert (recon.rows_in_file==recon[["before","after","inside_unmapped","mapped"]].sum(axis=1)).all(); recon.to_csv(OUT/"step3_mapping_reconciliation.csv"); display(recon)
mapped=ev[ev.mapped].copy(); mapped.to_csv(OUT/"step3_event_map_occurrences.csv",index=False)

boj=pd.read_csv(BOJ); boj["t"]=pd.to_datetime(boj.utc_datetime,utc=True,format="ISO8601",errors="coerce"); boj=boj.dropna(subset=["t"])
a=mapped[mapped.event.eq("BOJ Target Rate")].groupby(mapped[mapped.event.eq("BOJ Target Rate")].t.dt.tz_convert("Asia/Tokyo").dt.date).t.min(); b=boj.groupby(boj.t.dt.tz_convert("Asia/Tokyo").dt.date).t.min(); j=pd.concat({"g10":a,"calendar":b},axis=1).dropna()
def torow(x):
    q=x.astype("int64").to_numpy(); r=np.searchsorted(E,q,side="right"); ii=np.clip(r,0,len(E)-1); return np.where((r<len(E))&(S[ii]<=q),ii,-1)
same=torow(j.g10)==torow(j.calendar); lo=max(a.index.min(),b.index.min()); hi=min(a.index.max(),b.index.max())
bojcheck=pd.DataFrame([{"common_start":str(lo),"common_end":str(hi),"JST_dates_matched":len(j),"g10_missing_in_calendar":sum(lo<=d<=hi and d not in b.index for d in a.index),"calendar_missing_in_g10":sum(lo<=d<=hi and d not in a.index for d in b.index),"same_model_day":int(same.sum()),"total_compared":len(same),"median_timestamp_gap_minutes":float(((j.g10-j.calendar).dt.total_seconds()/60).median())}])
bojcheck.to_csv(OUT/"step3_boj_crosscheck.csv",index=False); display(bojcheck)

,rows_in_file,before,after,inside_unmapped,mapped,distinct_model_days
event,,,,,,
FOMC Rate Decision (Upper Bound),241,52,0,6,183,183
BOJ Target Rate,221,0,0,11,210,210
Change in Nonfarm Payrolls,354,77,0,9,268,267
CPI MoM,353,77,0,9,267,267
CPI YoY,284,7,0,9,268,268
Natl CPI YoY,298,20,0,9,269,269
Tokyo CPI Ex-Fresh Food YoY,298,20,0,8,270,270


,common_start,common_end,JST_dates_matched,g10_missing_in_calendar,calendar_missing_in_g10,same_model_day,total_compared,median_timestamp_gap_minutes
0,2015-01-21,2026-06-16,99,0,3,99,99,246.0


## Duplicate labels, independent hypotheses, and overlaps

US CPI MoM/YoY are collapsed only if their mapped model-day overlap is at least 98%. Japanese national and Tokyo CPI remain separate; their actual shared-day fraction is exported. “Clean event” means the session contains exactly one canonical target hypothesis, so duplicate labels inside the same canonical US CPI release do not create false overlap.

In [3]:
daysets={x:set(mapped.loc[mapped.event.eq(x),"panel_row"].astype(int)) for x in TARGETS}
def overlap(a,b): return len(daysets[a]&daysets[b])/min(len(daysets[a]),len(daysets[b]))
dup=pd.DataFrame([{"pair":"US CPI MoM vs YoY","shared_fraction":overlap("CPI MoM","CPI YoY")},{"pair":"JP national vs Tokyo CPI","shared_fraction":overlap("Natl CPI YoY","Tokyo CPI Ex-Fresh Food YoY")}]); dup.to_csv(OUT/"step3_duplicate_overlap_audit.csv",index=False); display(dup)
assert dup.loc[0,"shared_fraction"]>=.98 and .45<dup.loc[1,"shared_fraction"]<.80
canon={"FOMC Rate Decision (Upper Bound)":"FOMC","BOJ Target Rate":"BOJ","Change in Nonfarm Payrolls":"NFP","CPI MoM":"US CPI","CPI YoY":"US CPI","Natl CPI YoY":"JP national CPI","Tokyo CPI Ex-Fresh Food YoY":"Tokyo CPI"}
mapped["hypothesis"]=mapped.event.map(canon); hyp_days={h:sorted(mapped.loc[mapped.hypothesis.eq(h),"panel_row"].astype(int).unique()) for h in sorted(set(canon.values()))}
row_hyp={i:set() for i in range(len(panel))}
for h,rows in hyp_days.items():
    for i in rows: row_hyp[i].add(h)
event_day=np.array([bool(row_hyp[i]) for i in range(len(panel))]); n_hyp=np.array([len(row_hyp[i]) for i in range(len(panel))])
overlap_table=pd.DataFrame([{"hypothesis":h,"all_days":len(rows),"clean_days":sum(n_hyp[i]==1 for i in rows),"overlap_days":sum(n_hyp[i]>1 for i in rows)} for h,rows in hyp_days.items()]); overlap_table.to_csv(OUT/"step3_overlap_counts.csv",index=False); display(overlap_table)

,pair,shared_fraction
0,US CPI MoM vs YoY,1.000000
1,JP national vs Tokyo CPI,0.624535


,hypothesis,all_days,clean_days,overlap_days
0,BOJ,210,155,55
1,FOMC,183,158,25
2,JP national CPI,269,89,180
3,NFP,267,258,9
4,Tokyo CPI,270,95,175
5,US CPI,268,242,26


## Past-only cross-fitted positive counterfactuals

Models refit at each calendar-year boundary using only earlier uncontaminated eventless rows. This is historically implementable. The `lag_mask` version removes the event session plus all next `P` rows from estimation. The `recursive` version additionally replaces event-affected RV/IV states by their positive eventless predictions before those states can enter later lag windows.

In [4]:
STATES=["r2","rv","iv"]; BLOCKS={"rv":dec["eq2_blocks_retained"],"iv":dec["eq3_blocks_retained"]}; WD=pd.get_dummies(panel.model_day.dt.dayofweek,drop_first=True,dtype=float).to_numpy()
def ser(s): return panel.iv_quote if s=="iv" else panel[s]
def offs(src,tgt):
    if src=="iv" and tgt in ("rv","r2") and TIMING=="same_label": return list(range(0,P))
    return list(range(1,P+1))
def features(target,source_arrays=None):
    cols=[]
    for s in BLOCKS[target]:
      x=ser(s).to_numpy(float) if source_arrays is None else source_arrays[s]
      for k in offs(s,target): cols.append(np.r_[np.full(k,np.nan),x[:-k]] if k else x.copy())
    Xstate=np.log(np.maximum(np.column_stack(cols),1e-20))
    return np.column_stack([Xstate,WD]) if WEEKDAY else Xstate
contaminated=event_day.copy()
for k in range(1,P+1): contaminated |= np.r_[np.zeros(k,bool),event_day[:-k]]
eventless=~contaminated
models={}; pred_mask={t:np.full(len(panel),np.nan) for t in ("rv","iv")}
years=sorted(panel.model_day.dt.year.unique())
for target in ("rv","iv"):
    y=ser(target).to_numpy(float); X=features(target); usable=np.isfinite(y)&(y>0)&np.isfinite(X).all(1)
    for year in years:
        tr=usable&eventless&panel.model_day.lt(pd.Timestamp(f"{year}-01-01")).to_numpy(); te=usable&panel.model_day.dt.year.eq(year).to_numpy()
        if tr.sum()<MIN_TRAIN or not te.any(): continue
        m=sm.OLS(np.log(y[tr]),sm.add_constant(X[tr],has_constant="add")).fit(); models[(target,year)]=m; pred_mask[target][te]=np.exp(m.predict(sm.add_constant(X[te],has_constant="add")))
    assert np.nanmin(pred_mask[target])>0

# Recursive state replacement, using the same past-only year models.
state={"r2":panel.r2.to_numpy(float).copy(),"rv":panel.rv.to_numpy(float).copy(),"iv":panel.iv_quote.to_numpy(float).copy()}; pred_rec={t:np.full(len(panel),np.nan) for t in ("rv","iv")}
for i in range(len(panel)):
  year=int(panel.model_day.dt.year.iloc[i])
  for target in ("rv","iv"):
    m=models.get((target,year)); vals=[]
    if m is None: continue
    good=True
    for s in BLOCKS[target]:
      for k in offs(s,target):
        if i-k<0 or not np.isfinite(state[s][i-k]) or state[s][i-k]<=0: good=False; break
        vals.append(np.log(max(state[s][i-k],1e-20)))
      if not good: break
    if not good: continue
    if WEEKDAY: vals.extend(WD[i].tolist())
    pred_rec[target][i]=float(np.exp(m.predict(np.r_[1.,vals])[0])); assert pred_rec[target][i]>0
  if event_day[i]:
    for target in ("rv","iv"):
      if np.isfinite(pred_rec[target][i]): state[target][i]=pred_rec[target][i]
predout=pd.DataFrame({"model_day":panel.model_day,"event_day":event_day,"lag_contaminated":contaminated,"rv_mask":pred_mask["rv"],"iv_mask":pred_mask["iv"],"rv_recursive":pred_rec["rv"],"iv_recursive":pred_rec["iv"]})
predout.to_csv(OUT/"step3_crossfitted_counterfactuals.csv",index=False)
assert predout[["rv_mask","iv_mask","rv_recursive","iv_recursive"]].stack().gt(0).all()
print("event days",int(event_day.sum()),"lag-contaminated",int(contaminated.sum()),"eventless",int(eventless.sum()))

event days 1226 lag-contaminated 5164 eventless 678


## Literal weights, overlap sensitivity, empirical null, and primary inference

Group estimates are `exp(median(log omega_i))`, a conditional-median estimand centred on one under a correctly calibrated multiplicative null. CIs use circular moving blocks of five event occurrences. Primary p-values use circular shifts of event indicators over the complete daily log-ratio series, preserving serial structure; BH-FDR is applied across canonical event hypotheses and both equations.

In [5]:
def block_ci(x,B=N_BOOT,L=5):
    x=np.asarray(x,float); n=len(x)
    if n<10:return np.nan,np.nan
    out=[]
    for _ in range(B):
      starts=rng.integers(0,n,size=int(np.ceil(n/L))); z=np.concatenate([x[(s+np.arange(L))%n] for s in starts])[:n]; out.append(np.exp(np.median(z)))
    return tuple(np.percentile(out,[2.5,97.5]))
def shift_p(logratio,eventmask,obs,B=N_PERM):
    vals=[]; n=len(logratio)
    for k in rng.integers(P+1,n-P-1,size=B):
      m=np.roll(eventmask,k)&np.isfinite(logratio); vals.append(np.median(logratio[m]) if m.sum() else np.nan)
    vals=np.asarray(vals); return float((1+np.sum(np.abs(vals)>=abs(obs)))/(1+np.isfinite(vals).sum()))
rows=[]; occurrences=[]
for target in ("rv","iv"):
  actual=ser(target).to_numpy(float); cf=pred_mask[target]; lr=np.log(actual/cf); nullmask=eventless&np.isfinite(lr); null_med=float(np.exp(np.median(lr[nullmask])))
  for hyp,days in hyp_days.items():
    for clean in (False,True):
      rr=np.array([i for i in days if (not clean or n_hyp[i]==1)],int); rr=rr[np.isfinite(lr[rr])]; x=lr[rr]
      if len(x)<10: continue
      obs=float(np.median(x)); lo,hi=block_ci(x); em=np.zeros(len(panel),bool); em[rr]=True; pv=shift_p(lr,em,obs)
      rows.append({"target":target,"hypothesis":hyp,"sample":"clean" if clean else "all","n":len(x),"omega_pdf":float(np.exp(obs)),"ci_lo":lo,"ci_hi":hi,"p_circular_shift":pv,"eventless_null_median_omega":null_med,"omega_null_normalized":float(np.exp(obs)/null_med)})
      for i in rr: occurrences.append({"model_day":panel.model_day.iloc[i],"panel_row":i,"target":target,"hypothesis":hyp,"sample":"clean" if clean else "all","actual":actual[i],"counterfactual":cf[i],"omega_pdf":actual[i]/cf[i]})
primary=pd.DataFrame(rows); occ=pd.DataFrame(occurrences); allrows=primary[primary["sample"].eq("all")].copy(); allrows["q_BH_primary_all_event_12"]=multipletests(allrows.p_circular_shift,method="fdr_bh")[1]; primary=primary.merge(allrows[["target","hypothesis","q_BH_primary_all_event_12"]],on=["target","hypothesis"],how="left")
primary.to_csv(OUT/"step3_primary_event_weights.csv",index=False); occ.to_csv(OUT/"step3_occurrence_weights.csv",index=False); display(primary)
assert np.allclose(occ.omega_pdf,occ.actual/occ.counterfactual) and occ.counterfactual.gt(0).all()

,target,hypothesis,sample,n,omega_pdf,ci_lo,ci_hi,p_circular_shift,eventless_null_median_omega,omega_null_normalized,q_BH_primary_all_event_12
0,rv,BOJ,all,122,1.133067,0.916074,1.255671,0.208000,0.988477,1.146276,0.356571
1,rv,BOJ,clean,90,1.050262,0.876210,1.215139,0.668000,0.988477,1.062506,0.356571
2,rv,FOMC,all,102,1.421229,1.101794,1.901395,0.010000,0.988477,1.437798,0.030000
3,rv,FOMC,clean,86,1.301376,1.037090,1.846417,0.060000,0.988477,1.316547,0.030000
4,rv,JP national CPI,all,152,0.845657,0.752579,0.937950,0.116000,0.988477,0.855516,0.236000
5,rv,JP national CPI,clean,87,0.874837,0.655953,0.981414,0.311983,0.988477,0.885036,0.236000
6,rv,NFP,all,150,0.938157,0.848928,1.088029,0.516000,0.988477,0.949094,0.586909
7,rv,NFP,clean,145,0.962480,0.836475,1.095907,0.672000,0.988477,0.973700,0.586909
8,rv,Tokyo CPI,all,149,0.867245,0.776531,0.946042,0.118000,0.988477,0.877355,0.236000
9,rv,Tokyo CPI,clean,89,0.918946,0.777201,1.036667,0.533742,0.988477,0.929659,0.236000


## Recursive sensitivity and offset profiles

Offset 0 remains the only primary estimate. The offset profile is secondary. A max-statistic circular-shift p-value corrects any statement about the strongest of -1/0/+1; no uncorrected selected-offset p-value is reported.

In [6]:
sens=[]; profiles=[]
for target in ("rv","iv"):
  actual=ser(target).to_numpy(float)
  for variant,cf in [("lag_mask",pred_mask[target]),("recursive",pred_rec[target])]:
    lr=np.log(actual/cf)
    for hyp,days in hyp_days.items():
      rr=np.array(days,int); rr=rr[np.isfinite(lr[rr])]
      if len(rr)>=10: sens.append({"target":target,"hypothesis":hyp,"variant":variant,"n":len(rr),"omega_pdf":float(np.exp(np.median(lr[rr])))})
  lr=np.log(actual/pred_mask[target]); n=len(panel)
  for hyp,days in hyp_days.items():
    obs=[]
    for off in (-1,0,1):
      rr=np.array(days,int)+off; rr=rr[(rr>=0)&(rr<n)&np.isfinite(lr[rr])]; val=float(np.median(lr[rr])); obs.append(val); profiles.append({"target":target,"hypothesis":hyp,"offset":off,"n":len(rr),"omega_pdf":float(np.exp(val))})
    nullmax=[]; em=np.zeros(n,bool); em[np.array(days,int)]=True
    for k in rng.integers(P+1,n-P-1,size=N_PERM):
      sh=np.flatnonzero(np.roll(em,k)); vals=[]
      for off in (-1,0,1):
        rr=sh+off; rr=rr[(rr>=0)&(rr<n)]; rr=rr[np.isfinite(lr[rr])]; vals.append(abs(np.median(lr[rr])) if len(rr) else np.nan)
      nullmax.append(np.nanmax(vals))
    pmax=(1+np.sum(np.asarray(nullmax)>=max(abs(np.asarray(obs)))))/(1+len(nullmax))
    for r in profiles[-3:]: r["p_maxstat_offsets_FWER"]=float(pmax)
pd.DataFrame(sens).to_csv(OUT/"step3_counterfactual_sensitivity.csv",index=False); profile=pd.DataFrame(profiles); profile.to_csv(OUT/"step3_offset_profiles_secondary.csv",index=False); display(pd.DataFrame(sens)); display(profile)

,target,hypothesis,variant,n,omega_pdf
0,rv,BOJ,lag_mask,122,1.133067
1,rv,FOMC,lag_mask,102,1.421229
2,rv,JP national CPI,lag_mask,152,0.845657
3,rv,NFP,lag_mask,150,0.938157
4,rv,Tokyo CPI,lag_mask,149,0.867245
5,rv,US CPI,lag_mask,156,1.101809
6,rv,BOJ,recursive,123,1.144335
7,rv,FOMC,recursive,102,1.289871
8,rv,JP national CPI,recursive,152,0.871857
9,rv,NFP,recursive,151,0.962244


,target,hypothesis,offset,n,omega_pdf,p_maxstat_offsets_FWER
0,rv,BOJ,-1,120,0.983733,0.006
1,rv,BOJ,0,122,1.133067,0.006
2,rv,BOJ,1,120,0.785646,0.006
3,rv,FOMC,-1,103,0.912678,0.002
4,rv,FOMC,0,102,1.421229,0.002
5,rv,FOMC,1,102,1.185747,0.002
6,rv,JP national CPI,-1,153,1.084031,0.060
7,rv,JP national CPI,0,152,0.845657,0.060
8,rv,JP national CPI,1,148,1.017393,0.060
9,rv,NFP,-1,150,1.144589,0.144


## Surprise conditioning (secondary)

Pooled scheduled-event weights remain primary. Where `surprise_z` exists, the secondary regression relates log literal weight to absolute standardised surprise, with HAC uncertainty. It is descriptive and does not replace the pooled estimand.

In [7]:
sur=[]
for hyp in hyp_days:
  sub=mapped[mapped.hypothesis.eq(hyp)].sort_values("t").drop_duplicates("panel_row"); sub=sub[np.isfinite(sub.surprise_z)]
  for target in ("rv","iv"):
    a=ser(target).to_numpy(float); cf=pred_mask[target]; rr=sub.panel_row.astype(int).to_numpy(); ok=np.isfinite(a[rr])&np.isfinite(cf[rr])&(cf[rr]>0)
    if ok.sum()<20: continue
    y=np.log(a[rr[ok]]/cf[rr[ok]]); x=np.abs(sub.surprise_z.to_numpy(float)[ok]); m=sm.OLS(y,sm.add_constant(x)).fit(cov_type="HAC",cov_kwds={"maxlags":3})
    sur.append({"hypothesis":hyp,"target":target,"n":int(ok.sum()),"intercept":m.params[0],"abs_surprise_z_slope":m.params[1],"slope_se_HAC":m.bse[1],"slope_p_HAC":m.pvalues[1]})
sur=pd.DataFrame(sur); sur.to_csv(OUT/"step3_surprise_conditioning_secondary.csv",index=False); display(sur)

,hypothesis,target,n,intercept,abs_surprise_z_slope,slope_se_HAC,slope_p_HAC
0,BOJ,rv,20,0.087159,0.235084,0.044206,1.049336e-07
1,BOJ,iv,22,-0.242482,0.036916,0.020667,7.406413e-02
2,FOMC,rv,101,0.404448,-0.011279,0.072496,8.763671e-01
3,FOMC,iv,108,0.175833,0.021714,0.021622,3.152444e-01
4,JP national CPI,rv,152,-0.101560,-0.095661,0.105781,3.658219e-01
5,JP national CPI,iv,162,-0.057095,-0.055047,0.050052,2.714189e-01
6,NFP,rv,150,-0.069611,0.002973,0.001653,7.199800e-02
7,NFP,iv,161,-0.320442,0.005430,0.000774,2.254617e-12
8,Tokyo CPI,rv,149,-0.069914,0.002717,0.074213,9.707928e-01
9,Tokyo CPI,iv,162,-0.054190,-0.021165,0.047086,6.530761e-01


## Generated reconciliation and restrained conclusions

The summary below is generated from the current tables. FOMC timing, NFP damping, and every offset-dependent pattern remain secondary unless the exported max-statistic p-value survives. Predictive counterfactual departures are not causal effects: scheduled events may co-occur with regimes and unmeasured news.

In [8]:
pr=primary[primary["sample"].eq("all")].copy(); surviving=pr.loc[pr.q_BH_primary_all_event_12<ALPHA,["target","hypothesis","omega_pdf","q_BH_primary_all_event_12"]].to_dict("records")
summary={"seed":SEED,"inherits":{"split_date":dec["split_date"],"timing":TIMING,"lag":P,"eq2_blocks":BLOCKS["rv"],"eq3_blocks":BLOCKS["iv"]},
 "eventless_definition":f"usable sessions with no canonical target event and not in the next {P} model-day lag-contamination window",
 "primary_offset":0,"secondary_offsets":[-1,1],"literal_weight":"omega_i = actual_i / counterfactual_i","group_estimand":"exp(median(log omega_i))",
 "empirical_null_name":"omega_null_normalized (not the PDF weight)","canonical_hypotheses":sorted(hyp_days),"US_CPI_duplicate_fraction":float(dup.loc[0,'shared_fraction']),"JP_CPI_shared_fraction":float(dup.loc[1,'shared_fraction']),
 "BOJ_crosscheck":bojcheck.iloc[0].to_dict(),"all_counterfactuals_positive":bool(predout[["rv_mask","iv_mask","rv_recursive","iv_recursive"]].stack().gt(0).all()),
 "primary_FDR_survivors":surviving,"offset_findings_rule":"secondary; strongest-offset inference uses circular-shift max-statistic FWER","causal_claim":"none; conditional predictive departures only"}
(OUT/"step3_summary.json").write_text(json.dumps(summary,indent=2,default=str)); print(json.dumps(summary,indent=2,default=str))
assert summary["all_counterfactuals_positive"] and dup.loc[0,"shared_fraction"]>=.98 and dup.loc[1,"shared_fraction"]<.80

{
  "seed": 16017,
  "inherits": {
    "split_date": "2022-04-12",
    "timing": "advance",
    "lag": 5,
    "eq2_blocks": [
      "iv",
      "rv"
    ],
    "eq3_blocks": [
      "iv"
    ]
  },
  "eventless_definition": "usable sessions with no canonical target event and not in the next 5 model-day lag-contamination window",
  "primary_offset": 0,
  "secondary_offsets": [
    -1,
    1
  ],
  "literal_weight": "omega_i = actual_i / counterfactual_i",
  "group_estimand": "exp(median(log omega_i))",
  "empirical_null_name": "omega_null_normalized (not the PDF weight)",
  "canonical_hypotheses": [
    "BOJ",
    "FOMC",
    "JP national CPI",
    "NFP",
    "Tokyo CPI",
    "US CPI"
  ],
  "US_CPI_duplicate_fraction": 1.0,
  "JP_CPI_shared_fraction": 0.6245353159851301,
  "BOJ_crosscheck": {
    "common_start": "2015-01-21",
    "common_end": "2026-06-16",
    "JST_dates_matched": 99,
    "g10_missing_in_calendar": 0,
    "calendar_missing_in_g10": 3,
    "same_model_day": 99,
    "to